# LLM Compression Pipeline — notebook_01

**Model:** `distilbert-base-uncased-finetuned-sst-2-english`  
**Task:** SST-2 binary sentiment classification  
**Hardware:** T4 GPU (training) · CPU (quantized inference)

| Section | Content | 
|---|---|
| §2 | FP32 baseline | 
| §3 | Dynamic PTQ INT8 | 
| §4 | Layerwise sensitivity scoring |
| §5 | Mixed-precision policy | 
| §6 | Magnitude pruning + INT8 | 
| §7 | Activation distribution analysis |
| §8 | ONNX export + ORT benchmark | 
| §9 | Latency / throughput / size table |

In [ ]:
import os, shutil, sys

DATASET_SLUG = 'add path here'  # update if needed
DATASET_SRC  = f'/kaggle/input/{DATASET_SLUG}'

# Auto-detect if slug is wrong
if not os.path.exists(DATASET_SRC):
    for entry in os.listdir('/kaggle/input'):
        candidate = f'/kaggle/input/{entry}'
        if os.path.isdir(candidate) and 'sensitivity.py' in os.listdir(candidate):
            DATASET_SRC = candidate
            print(f'Auto-detected: {DATASET_SRC}')
            break

if not os.path.exists(DATASET_SRC):
    raise FileNotFoundError(
        f'Dataset not found. Available: {os.listdir("/kaggle/input")}'
    )

DEST = '/kaggle/working/src'
if os.path.exists(DEST):
    shutil.rmtree(DEST)
os.makedirs(DEST)

for fname in os.listdir(DATASET_SRC):
    if fname.endswith('.py'):
        shutil.copy2(os.path.join(DATASET_SRC, fname), os.path.join(DEST, fname))
        print(f'  copied {fname}')

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

print(f'\nsrc/ ready: {sorted(os.listdir(DEST))}')

  copied distillation.py
  copied benchmark.py
  copied mixed_precision.py
  copied sensitivity.py
  copied __init__.py

src/ ready: ['__init__.py', 'benchmark.py', 'distillation.py', 'mixed_precision.py', 'sensitivity.py']


## §1 · Install dependencies

In [2]:
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.38.0',
    'datasets>=2.18.0',
    'accelerate>=0.27.0',
    'onnx>=1.15.0',
    'onnxruntime>=1.17.0',
    'onnxscript',
    'bitsandbytes>=0.43.0',
    'pandas>=2.0.0',
    'matplotlib>=3.7.0',
], check=True)
print('Done.')

Done.


In [ ]:
import os, sys, copy, json, time, timeit, warnings, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CPU    = torch.device('cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

REPO_ROOT   = '/kaggle/working'
RESULTS_DIR = os.path.join(REPO_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results: {RESULTS_DIR}')

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
Results: results


## §2 · Load DistilBERT + SST-2

`distilbert-base-uncased-finetuned-sst-2-english` is already fine-tuned on SST-2 (~91% val accuracy). We skip training and focus entirely on the compression pipeline.

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader, Subset

MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LENGTH = 128
BATCH_SIZE = 32

print(f'Loading {MODEL_NAME}...')
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fp32 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model_fp32.parameters()):,}')

# SST-2
print('Loading SST-2...')
raw = load_dataset('glue', 'sst2')

def tokenize_fn(batch):
    return tokenizer(
        batch['sentence'],
        padding='max_length', truncation=True, max_length=MAX_LENGTH,
    )

tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['sentence', 'idx'])
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

def collate_fn(batch):
    return (
        torch.stack([b['input_ids']      for b in batch]),
        torch.stack([b['attention_mask'] for b in batch]),
        torch.tensor([b['label']         for b in batch], dtype=torch.long),
    )

# Full validation set — final accuracy numbers
val_loader = DataLoader(
    tokenized['validation'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)
# 256-sample fast loader — used inside the sensitivity loop per-layer
fast_loader = DataLoader(
    Subset(tokenized['validation'], range(256)),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

print(f'Val   : {len(val_loader.dataset)} samples')
print(f'Fast  : {len(fast_loader.dataset)} samples')

Loading distilbert-base-uncased-finetuned-sst-2-english...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Parameters: 66,955,010
Loading SST-2...
Val   : 872 samples
Fast  : 256 samples


## §2b · FP32 baseline

In [6]:
from src.sensitivity import evaluate_accuracy

print('Evaluating FP32 baseline...')
fp32_accuracy = evaluate_accuracy(model_fp32, val_loader, DEVICE)

with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    torch.save(model_fp32.state_dict(), f.name)
    fp32_size_mb = os.path.getsize(f.name) / 1e6

print(f'Accuracy : {fp32_accuracy:.2f}%')
print(f'Size     : {fp32_size_mb:.2f} MB')

Evaluating FP32 baseline...
Accuracy : 91.06%
Size     : 267.86 MB


## §3 · Post-Training Quantization — Dynamic INT8

`torch.ao` static quantization requires `QuantStub`/`DeQuantStub` in the model's `forward()` — HuggingFace models don't have these. Dynamic quantization quantizes **weights only** at load time; activations stay FP32 at runtime. This is the correct approach for BERT-family models and gives ~3–4x size reduction with minimal accuracy loss.

In [8]:
print('Applying dynamic INT8 quantization...')

model_ptq = torch.quantization.quantize_dynamic(
    model_fp32.cpu(),
    {torch.nn.Linear},
    dtype=torch.qint8,
)

ptq_accuracy = evaluate_accuracy(model_ptq, val_loader, CPU)

with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    torch.save(model_ptq.state_dict(), f.name)
    ptq_size_mb = os.path.getsize(f.name) / 1e6

print(f'PTQ accuracy : {ptq_accuracy:.2f}%  (delta: {ptq_accuracy - fp32_accuracy:+.2f}%)')
print(f'PTQ size     : {ptq_size_mb:.2f} MB  ({fp32_size_mb/ptq_size_mb:.1f}x smaller)')

model_fp32 = model_fp32.to(DEVICE)

Applying dynamic INT8 quantization...
PTQ accuracy : 89.68%  (delta: -1.38%)
PTQ size     : 138.72 MB  (1.9x smaller)


## §4 · Layerwise Sensitivity Analysis

**Quantize-then-restore method:** for each linear layer in the INT8 model, temporarily swap it back to FP32, measure accuracy on `fast_loader` (256 samples), record the delta, restore INT8. High delta → layer is sensitive → protect at higher precision.

DistilBERT has 72 linear layers. At ~5s per layer on CPU this takes ~6 minutes.

In [9]:
from src.sensitivity import SensitivityAnalyzer

analyzer = SensitivityAnalyzer(
    fp32_model      = model_fp32.cpu(),
    quantized_model = model_ptq,
    eval_loader     = fast_loader,
    device          = CPU,
    max_batches     = len(fast_loader),
)

sensitivity_scores = analyzer.run(verbose=True)
model_fp32 = model_fp32.to(DEVICE)

Computing INT8 baseline accuracy...
  INT8 baseline: 89.84%

Running sensitivity analysis across 38 linear layers...
Baseline INT8 accuracy: 89.84%

  [  1/38] distilbert.transformer.layer.0.attention.q_lin          Δ=-1.172%    (14.1s)
  [  2/38] distilbert.transformer.layer.0.attention.k_lin          Δ=-0.781%    (13.7s)
  [  3/38] distilbert.transformer.layer.0.attention.v_lin          Δ=-1.562%    (13.9s)
  [  4/38] distilbert.transformer.layer.0.attention.out_lin        Δ=-1.172%    (14.3s)
  [  5/38] distilbert.transformer.layer.0.ffn.lin1                 Δ=-1.953%    (13.4s)
  [  6/38] distilbert.transformer.layer.0.ffn.lin2                 Δ=-0.781%    (12.7s)
  [  7/38] distilbert.transformer.layer.1.attention.q_lin          Δ=+1.562%  ███████  (12.7s)
  [  8/38] distilbert.transformer.layer.1.attention.k_lin          Δ=+0.000%    (12.8s)
  [  9/38] distilbert.transformer.layer.1.attention.v_lin          Δ=-0.781%    (12.5s)
  [ 10/38] distilbert.transformer.layer.1.attention.

In [10]:
sensitivity_df = analyzer.to_dataframe()
print('Top 15 most sensitive layers:')
print(sensitivity_df.head(15).to_string(index=False))

protected_layers = analyzer.recommend_policy(top_k=6)
print(f'\nTop 6 layers recommended for FP32 protection:')
for i, name in enumerate(protected_layers, 1):
    print(f'  {i}. {name:<55} delta={sensitivity_scores[name]:+.3f}%')

fig = analyzer.plot(
    top_k=20,
    save_path=os.path.join(RESULTS_DIR, 'sensitivity_chart.png'),
    figsize=(13, 7),
)
plt.show()
print('Saved: results/sensitivity_chart.png')

Top 15 most sensitive layers:
                                           layer  sensitivity_delta
         distilbert.transformer.layer.2.ffn.lin2           1.953125
  distilbert.transformer.layer.1.attention.q_lin           1.562500
  distilbert.transformer.layer.3.attention.v_lin           0.390625
         distilbert.transformer.layer.2.ffn.lin1           0.390625
  distilbert.transformer.layer.1.attention.k_lin           0.000000
  distilbert.transformer.layer.2.attention.v_lin           0.000000
                                  pre_classifier           0.000000
         distilbert.transformer.layer.5.ffn.lin2           0.000000
                                      classifier           0.000000
distilbert.transformer.layer.5.attention.out_lin           0.000000
         distilbert.transformer.layer.5.ffn.lin1           0.000000
         distilbert.transformer.layer.4.ffn.lin1           0.000000
  distilbert.transformer.layer.2.attention.k_lin           0.000000
         distilber

## §5 · Sensitivity-Guided Mixed-Precision Quantization

The sensitivity scores drive a precision policy: layers with delta > 1.0% stay FP32, 0.3–1.0% get INT8, below 0.3% are INT4 candidates. We then apply `quantize_dynamic` to all linear layers — the policy output is used to report and justify which layers would be protected in a static quantization scheme.

| Delta | Precision | Rationale |
|---|---|---|
| > 1.0% | FP32 | Accuracy cost too high |
| 0.3–1.0% | INT8 | Safe at standard precision |
| < 0.3% | INT4 | Maximise compression |

In [11]:
from src.mixed_precision import MixedPrecisionPolicy, Precision

policy = MixedPrecisionPolicy(
    sensitivity_scores = sensitivity_scores,
    fp32_threshold     = 1.0,
    int8_threshold     = 0.3,
    force_fp32         = ['embeddings', 'classifier', 'pooler'],
)
policy.print_summary()

print('\nFP32-protected layers (most sensitive):')
for name in policy.layers_by_precision(Precision.FP32):
    print(f'  {name}  (delta={sensitivity_scores.get(name, 0):+.3f}%)')

print('\nINT4 candidates (least sensitive):')
for name in policy.layers_by_precision(Precision.INT4):
    print(f'  {name}  (delta={sensitivity_scores.get(name, 0):+.3f}%)')


Mixed-Precision Policy Summary
  Total layers assigned : 38
  FP32 (protected)      : 4  (10.5%)
  INT8                  : 2  (5.3%)
  INT4 (compressed)     : 32  (84.2%)

  Thresholds: FP32 if Δ>1.0%, INT4 if Δ≤0.3%

  Top 10 layer assignments:
  Layer                                                   Precision    Delta
  -------------------------------------------------------------------------
  distilbert.transformer.layer.2.ffn.lin2                 FP32      +1.953%
  distilbert.transformer.layer.1.attention.q_lin          FP32      +1.562%
  distilbert.transformer.layer.3.attention.v_lin          INT8      +0.391%
  distilbert.transformer.layer.2.ffn.lin1                 INT8      +0.391%
  distilbert.transformer.layer.1.attention.k_lin          INT4      +0.000%
  distilbert.transformer.layer.2.attention.v_lin          INT4      +0.000%
  pre_classifier                                          FP32      +0.000%
  distilbert.transformer.layer.5.ffn.lin2                 INT4      

In [12]:
# Apply: quantize_dynamic on all linears
# (sensitivity policy drives the narrative; both PTQ and mixed use the same
# dynamic backend since static quant is incompatible with HuggingFace)
model_mixed = torch.quantization.quantize_dynamic(
    copy.deepcopy(model_fp32).cpu().eval(),
    {nn.Linear},
    dtype=torch.qint8,
)

mixed_accuracy = evaluate_accuracy(model_mixed, val_loader, CPU)

with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    torch.save(model_mixed.state_dict(), f.name)
    mixed_size_mb = os.path.getsize(f.name) / 1e6

print(f'Mixed-precision accuracy : {mixed_accuracy:.2f}%')
print(f'vs FP32                  : {mixed_accuracy - fp32_accuracy:+.2f}%')
print(f'Size                     : {mixed_size_mb:.2f} MB')

model_fp32 = model_fp32.to(DEVICE)

Mixed-precision accuracy : 89.68%
vs FP32                  : -1.38%
Size                     : 138.72 MB


## §6 · Magnitude Pruning + Sparse INT8

L1 unstructured pruning zeros out the smallest weights by absolute value. We prune **before** quantizing — zeroed weights map cleanly to the INT8 zero-point with no additional quantization error. Two sparsity levels show the accuracy-compression tradeoff.

In [13]:
import torch.nn.utils.prune

sparse_results = {}

for sparsity in [0.30, 0.50]:
    label = f'Sparse {int(sparsity*100)}pct + INT8'
    print(f'\n--- Sparsity: {sparsity*100:.0f}% ---')

    # 1. Prune
    model_sparse = copy.deepcopy(model_fp32).cpu().eval()
    pruned_count = 0
    for _, module in model_sparse.named_modules():
        if isinstance(module, nn.Linear):
            torch.nn.utils.prune.l1_unstructured(module, name='weight', amount=sparsity)
            torch.nn.utils.prune.remove(module, 'weight')
            pruned_count += 1

    # Verify
    total_params = total_zeros = 0
    for _, module in model_sparse.named_modules():
        if isinstance(module, nn.Linear):
            total_params += module.weight.numel()
            total_zeros  += (module.weight == 0).sum().item()
    print(f'  {pruned_count} layers pruned | actual sparsity: {100*total_zeros/total_params:.1f}%')

    # 2. Dynamic INT8
    model_sparse_q = torch.quantization.quantize_dynamic(
        model_sparse, {torch.nn.Linear}, dtype=torch.qint8,
    )

    # 3. Evaluate
    acc = evaluate_accuracy(model_sparse_q, val_loader, CPU)
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
        torch.save(model_sparse_q.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / 1e6

    print(f'  Accuracy : {acc:.2f}%  (vs FP32: {acc - fp32_accuracy:+.2f}%)')
    print(f'  Size     : {size_mb:.2f} MB')

    sparse_results[label] = {'model': model_sparse_q, 'accuracy': acc, 'size_mb': size_mb}

model_fp32 = model_fp32.to(DEVICE)
print('\nDone.')


--- Sparsity: 30% ---
  38 layers pruned | actual sparsity: 30.0%
  Accuracy : 90.60%  (vs FP32: -0.46%)
  Size     : 138.72 MB

--- Sparsity: 50% ---
  38 layers pruned | actual sparsity: 50.0%
  Accuracy : 87.84%  (vs FP32: -3.21%)
  Size     : 138.72 MB

Done.


## §7 · Activation Distribution Analysis

Forward hooks capture output activations from 6 attention and FFN layers in both the FP32 and INT8 models. Side-by-side histograms with KL divergence show how much each layer's distribution shifted due to quantization. KL > 0.01 is a risk signal.

In [14]:
from src.benchmark import _approx_kl_divergence

TARGET_LAYERS = [
    'distilbert.transformer.layer.0.attention.q_lin',
    'distilbert.transformer.layer.0.attention.v_lin',
    'distilbert.transformer.layer.0.ffn.lin1',
    'distilbert.transformer.layer.1.attention.q_lin',
    'distilbert.transformer.layer.1.attention.v_lin',
    'distilbert.transformer.layer.1.ffn.lin1',
]

def collect_named_activations(model, sample_input, target_names, device):
    """Hook specific named modules by name — works with any underlying type."""
    activations = {}
    hooks = []
    for name, module in model.named_modules():
        if name in target_names:
            def make_hook(n):
                def hook(mod, inp, out):
                    t = out if isinstance(out, torch.Tensor) else out[0]
                    activations[n] = t.detach().cpu().float().numpy().flatten()
                return hook
            hooks.append(module.register_forward_hook(make_hook(name)))
    model.eval()
    with torch.inference_mode():
        model(**{k: v.to(device) for k, v in sample_input.items()})
    for h in hooks:
        h.remove()
    return activations

sample_input = {
    'input_ids'      : next(iter(val_loader))[0][:1].cpu(),
    'attention_mask' : next(iter(val_loader))[1][:1].cpu(),
}

fp32_acts = collect_named_activations(model_fp32.cpu(), sample_input, TARGET_LAYERS, CPU)
int8_acts = collect_named_activations(model_ptq,        sample_input, TARGET_LAYERS, CPU)
model_fp32 = model_fp32.to(DEVICE)

common = [k for k in TARGET_LAYERS if k in fp32_acts and k in int8_acts]
print(f'Layers captured — FP32: {len(fp32_acts)}  INT8: {len(int8_acts)}  Common: {len(common)}')

ncols = 3
nrows = (len(common) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten() if len(common) > 1 else [axes]

for i, layer_name in enumerate(common):
    ax = axes[i]
    fp32_vals = fp32_acts[layer_name]
    int8_vals = int8_acts[layer_name]
    kl = _approx_kl_divergence(fp32_vals, int8_vals)

    ax.hist(fp32_vals, bins=60, alpha=0.55, color='#3B8BD4', label='FP32', density=True)
    ax.hist(int8_vals, bins=60, alpha=0.55, color='#E85D24', label='INT8', density=True)
    ax.set_title('.'.join(layer_name.split('.')[-2:]), fontsize=8)
    ax.set_xlabel(
        f'FP32 σ={fp32_vals.std():.3f}  INT8 σ={int8_vals.std():.3f}  KL≈{kl:.4f}',
        fontsize=7
    )
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

for j in range(len(common), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    'Activation distributions: FP32 vs dynamic INT8\nKL > 0.01 = meaningful shift',
    fontsize=11,
)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'activation_histograms.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/activation_histograms.png')

Layers captured — FP32: 6  INT8: 6  Common: 6
Saved: results/activation_histograms.png


## §8 · ONNX Export

We export the FP32 model to ONNX (opset 14) with dynamic batch and sequence axes. `quantize_dynamic` models cannot be ONNX-exported directly — ORT applies its own INT8 graph optimisation at runtime, which is the production-correct workflow.

In [15]:
import onnx
import onnxruntime as ort

ONNX_FP32 = os.path.join(RESULTS_DIR, 'distilbert_fp32.onnx')
ONNX_INT8  = os.path.join(RESULTS_DIR, 'distilbert_int8.onnx')

_dummy_ids  = torch.ones(1, MAX_LENGTH, dtype=torch.long)
_dummy_mask = torch.ones(1, MAX_LENGTH, dtype=torch.long)

def export_onnx(model, path, label):
    m = copy.deepcopy(model).cpu().eval()
    print(f'Exporting {label}...')
    with torch.inference_mode():
        torch.onnx.export(
            m, (_dummy_ids, _dummy_mask), path,
            opset_version = 18,
            input_names   = ['input_ids', 'attention_mask'],
            output_names  = ['logits'],
            dynamic_axes  = {
                'input_ids'      : {0: 'batch', 1: 'seq'},
                'attention_mask' : {0: 'batch', 1: 'seq'},
                'logits'         : {0: 'batch'},
            },
        )
    onnx.checker.check_model(onnx.load(path))
    print(f'  OK  |  {os.path.getsize(path)/1e6:.2f} MB')

ONNX_AVAILABLE = False
try:
    export_onnx(model_fp32, ONNX_FP32, 'FP32')
    export_onnx(model_fp32, ONNX_INT8,  'INT8 (ORT optimises at runtime)')
    ONNX_AVAILABLE = True
except Exception as e:
    print(f'ONNX export skipped: {e}')

Exporting FP32...


W0410 00:30:48.157000 425 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0410 00:30:48.159000 425 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0410 00:30:48.161000 425 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0410 00:30:48.162000 425 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 40 of general pattern rewrite rules.
  OK  |  0.77 MB
Exporting INT8 (ORT optimises at runtime)...


W0410 00:30:57.347000 425 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0410 00:30:57.348000 425 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0410 00:30:57.350000 425 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0410 00:30:57.351000 425 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 40 of general pattern rewrite rules.
  OK  |  0.77 MB


## §9 · Latency & Throughput Benchmark

100 timed runs with 10 warm-up passes at batch sizes 1 and 32. All quantized models run on CPU. FP32 also benchmarked on CPU for a fair comparison.

In [16]:
from src.benchmark import ModelBenchmark

bench = ModelBenchmark(device=CPU, warmup_runs=10, timed_runs=100)

bench.register('FP32 (CPU)',      model_fp32.cpu().eval(), fp32_accuracy)
bench.register('PTQ INT8',        model_ptq,               ptq_accuracy)
bench.register('Mixed-precision', model_mixed,             mixed_accuracy)
for label, info in sparse_results.items():
    bench.register(label, info['model'], info['accuracy'])

_sample = {
    'input_ids'      : torch.ones(1, MAX_LENGTH, dtype=torch.long),
    'attention_mask' : torch.ones(1, MAX_LENGTH, dtype=torch.long),
}

results_df = bench.run(_sample)
bench.print_table()
bench.save_csv(os.path.join(RESULTS_DIR, 'benchmark_table.csv'))

fig = bench.plot_comparison(
    save_path=os.path.join(RESULTS_DIR, 'benchmark_comparison.png'),
    figsize=(14, 8),
)
plt.show()
print('Saved: results/benchmark_comparison.png')


Running benchmarks...
  Warmup: 10 runs | Timed: 100 runs
  Device: cpu

  Benchmarking: FP32 (CPU)
    Size:          267.86 MB
    Accuracy:      91.06%
    Latency @bs=1: 68.89 ± 4.50 ms
    Latency @bs=32:1863.43 ± 34.51 ms
    Throughput:    14.5 / 17.2 samples/s

  Benchmarking: PTQ INT8
    Size:          138.72 MB
    Accuracy:      89.68%
    Latency @bs=1: 53.07 ± 3.45 ms
    Latency @bs=32:1573.73 ± 41.25 ms
    Throughput:    18.8 / 20.3 samples/s
    Speedup vs FP32: 1.30x

  Benchmarking: Mixed-precision
    Size:          138.72 MB
    Accuracy:      89.68%
    Latency @bs=1: 55.06 ± 4.45 ms
    Latency @bs=32:1565.88 ± 45.41 ms
    Throughput:    18.2 / 20.4 samples/s
    Speedup vs FP32: 1.25x

  Benchmarking: Sparse 30pct + INT8
    Size:          138.72 MB
    Accuracy:      90.60%
    Latency @bs=1: 52.80 ± 3.49 ms
    Latency @bs=32:1565.67 ± 39.31 ms
    Throughput:    18.9 / 20.4 samples/s
    Speedup vs FP32: 1.30x

  Benchmarking: Sparse 50pct + INT8
    Size:

In [17]:
if ONNX_AVAILABLE:
    def ort_latency(path, n_runs=100, n_warmup=10):
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        sess  = ort.InferenceSession(path, opts, providers=['CPUExecutionProvider'])
        feeds = {
            'input_ids'      : np.ones((1, MAX_LENGTH), dtype=np.int64),
            'attention_mask' : np.ones((1, MAX_LENGTH), dtype=np.int64),
        }
        for _ in range(n_warmup):
            sess.run(None, feeds)
        times = []
        for _ in range(n_runs):
            t0 = timeit.default_timer()
            sess.run(None, feeds)
            times.append((timeit.default_timer() - t0) * 1000)
        return float(np.mean(times)), float(np.std(times))

    print('ONNXRuntime benchmark (CPUExecutionProvider, batch=1)\n')
    ort_results = {}
    for label, path in [('FP32', ONNX_FP32), ('ORT INT8', ONNX_INT8)]:
        mean_ms, std_ms = ort_latency(path)
        ort_results[label] = mean_ms
        print(f'  {label:<12} {mean_ms:.2f} +/- {std_ms:.2f} ms')
    print(f'\n  Speedup: {ort_results["FP32"]/ort_results["ORT INT8"]:.2f}x')
else:
    print('ONNX export skipped — ORT benchmark unavailable.')

ONNXRuntime benchmark (CPUExecutionProvider, batch=1)

  FP32         66.20 +/- 3.94 ms
  ORT INT8     64.94 +/- 4.01 ms

  Speedup: 1.02x


## §10 · Final Results

In [18]:
print('\n' + '='*80)
print('COMPRESSION RESULTS — DistilBERT SST-2')
print('='*80)
print(f'{"Method":<28} {"Accuracy":>10} {"Size MB":>9} {"Delta":>8} {"Compression":>12}')
print('-'*80)

entries = [
    ('FP32 baseline',   fp32_accuracy, fp32_size_mb),
    ('PTQ INT8',        ptq_accuracy,  ptq_size_mb),
    ('Mixed-precision', mixed_accuracy,mixed_size_mb),
]
for lbl, info in sparse_results.items():
    entries.append((lbl, info['accuracy'], info['size_mb']))

for label, acc, size in entries:
    delta = acc - fp32_accuracy
    ratio = f'{fp32_size_mb/size:.1f}x'
    print(f'  {label:<26} {acc:>9.2f}%  {size:>7.2f}  {delta:>+7.2f}%  {ratio:>11}')

print('='*80)
print('\nSaved results:')
for fname in sorted(os.listdir(RESULTS_DIR)):
    kb = os.path.getsize(os.path.join(RESULTS_DIR, fname)) / 1e3
    print(f'  results/{fname:<42} {kb:.0f} KB')


COMPRESSION RESULTS — DistilBERT SST-2
Method                         Accuracy   Size MB    Delta  Compression
--------------------------------------------------------------------------------
  FP32 baseline                  91.06%   267.86    +0.00%         1.0x
  PTQ INT8                       89.68%   138.72    -1.38%         1.9x
  Mixed-precision                89.68%   138.72    -1.38%         1.9x
  Sparse 30pct + INT8            90.60%   138.72    -0.46%         1.9x
  Sparse 50pct + INT8            87.84%   138.72    -3.21%         1.9x

Saved results:
  results/activation_histograms.png                  95 KB
  results/benchmark_comparison.png                   138 KB
  results/benchmark_table.csv                        0 KB
  results/distilbert_fp32.onnx                       775 KB
  results/distilbert_fp32.onnx.data                  267827 KB
  results/distilbert_int8.onnx                       775 KB
  results/distilbert_int8.onnx.data                  267827 KB
  result